In [2]:
import pandas as pd
import numpy as np
import cv2
import librosa
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: Data Ingestion
def load_metadata(metadata_path):
    """Load metadata from an Excel file."""
    return pd.read_excel(metadata_path)

def load_video(video_path):
    """Load video file."""
    return cv2.VideoCapture(video_path)

def load_audio(audio_path):
    """Load audio file."""
    return librosa.load(audio_path, sr=None)

# Step 2: Feature Extraction
def extract_video_features(video_path):
    """Extract features from video."""
    cap = load_video(video_path)
    frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        # Example feature: frame mean color
        frame_mean = cv2.mean(frame)[:3]
        frames.append(frame_mean)
    cap.release()
    return np.mean(frames, axis=0)  # Return average color features

def extract_audio_features(audio_path):
    """Extract features from audio."""
    y, sr = load_audio(audio_path)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    return np.mean(mfccs, axis=1)  # Return average MFCCs

# Step 3: Feature Alignment
def align_features(metadata, video_dir, audio_dir):
    """Align features with metadata."""
    features = []
    labels = []
    for index, row in metadata.iterrows():
        video_path = os.path.join(video_dir, row['VideoID'] + '.mp4')
        audio_path = os.path.join(audio_dir, row['VideoID'] + '.wav')
        
        video_features = extract_video_features(video_path)
        audio_features = extract_audio_features(audio_path)
        
        combined_features = np.concatenate((video_features, audio_features))
        features.append(combined_features)
        labels.append(row['HowFunny'])  # Assuming 'how_funny' is the label column

    return np.array(features), np.array(labels)

# Step 4: Model Development and Training
def train_model(X, y):
    """Train a machine learning model."""
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    return model, X_test, y_test

# Step 5: Evaluation
def evaluate_model(model, X_test, y_test):
    """Evaluate the trained model."""
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

# Main pipeline
def main(metadata_path, video_dir, audio_dir):
    metadata = load_metadata(metadata_path)
    features, labels = align_features(metadata, video_dir, audio_dir)
    model, X_test, y_test = train_model(features, labels)
    evaluate_model(model, X_test, y_test)

# Example usage
if __name__ == "__main__":
    excel_path = 'data/excel.xlsx'
    video_path = 'data/videos'
    audio_path = video_path

    main(excel_path, video_path, audio_path)

OpenCV: Couldn't read video stream from file "data/videos/NFCK3A.joke1.rep1.take1.ThatsNotAHat.mp4.mp4"
/opt/anaconda3/envs/videop/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3432: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/anaconda3/envs/videop/lib/python3.11/site-packages/numpy/core/_methods.py:190: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/var/folders/fp/nfh3jlt16mv7dm_xv5cq2c040000gn/T/ipykernel_7633/172650895.py:23: UserWarning: PySoundFile failed. Trying audioread instead.
  return librosa.load(audio_path, sr=None)
/opt/anaconda3/envs/videop/lib/python3.11/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


FileNotFoundError: [Errno 2] No such file or directory: 'data/videos/NFCK3A.joke1.rep1.take1.ThatsNotAHat.mp4.wav'

In [9]:
import pandas as pd
import numpy as np
import cv2
import librosa
import os
from moviepy.editor import VideoFileClip
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: Data Ingestion
def load_metadata(metadata_path):
    """Load metadata from an Excel file."""
    try:
        metadata = pd.read_excel(metadata_path)
        print(f"Metadata loaded successfully from {metadata_path}.")
        return metadata
    except Exception as e:
        print(f"Error loading metadata: {e}")
        raise

def extract_audio_from_video(video_path):
    """Extract audio from a video file."""
    if not os.path.exists(video_path):
        print(f"Error: Video file not found: {video_path}")
        return None
    try:
        video = VideoFileClip(video_path)
        audio = video.audio
        return audio.to_soundarray(fps=22000)  # Extract audio as an array
    except Exception as e:
        print(f"Error processing video for {video_path}: {e}")
        return None

# Step 2: Feature Extraction
def align_features(metadata, video_dir):
    """Align features and labels from metadata."""
    features = []
    labels = []
    for index, row in metadata.iterrows():
        video_filename = row['VideoID']  # Assuming this column exists
        video_path = os.path.join(video_dir, video_filename)

        # Check if the video file exists
        if not os.path.isfile(video_path):
            print(f"Error: Could not open video file {video_path}.")
            continue  # Skip to the next file if it doesn't exist

        # Extract audio features
        audio_features = extract_audio_from_video(video_path)
        if audio_features is None:
            continue  # Skip if audio extraction failed

        # Assuming labels are in a column named 'label'
        label = row['HowFunny']
        features.append(audio_features)
        labels.append(label)

    return np.array(features), np.array(labels)

# Step 3: Model Training
def train_model(features, labels):
    """Train a Random Forest model."""
    X_train, X_test, y_train, y_test = train_test_split(features, labels,
                                                         test_size=0.2,
                                                           random_state=42)
    model = RandomForestClassifier()
    model.fit(X_train, y_train)
    return model, X_test, y_test

# Step 4: Evaluation
def evaluate_model(model, X_test, y_test):
    """Evaluate the trained model."""
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

# Main pipeline
def main(metadata_path, video_dir):
    metadata = load_metadata(metadata_path)
    features, labels = align_features(metadata, video_dir)
    
    if features.size == 0 or labels.size == 0:
        print("No features or labels to train the model.")
        return
    
    model, X_test, y_test = train_model(features, labels)
    evaluate_model(model, X_test, y_test)

# Example usage
if __name__ == "__main__":
    main('../data/excel.xlsx', '../data/videos/')

Metadata loaded successfully from ../data/excel.xlsx.
Error: Could not open video file ../data/videos/2UWdXP.joke3.rep1.take1.ThatsNotAHat.mp4.
Error processing video for ../data/videos/ZRSRdA.joke3.rep3.take1.NomNomNom.mp4: 'NoneType' object has no attribute 'to_soundarray'


ValueError: setting an array element with a sequence.